# BlueField DPUs
BlueField is a next-generation Data Processing Unit (DPU) developed by NVIDIA (formerly Mellanox Technologies). It integrates powerful computing, networking, and storage acceleration capabilities into a single device. Designed for modern data centers, BlueField DPUs offer hardware-accelerated data processing, efficient resource management, and enhanced security features.

## Key Features of BlueFields
- Integrated Compute and Networking:
  - Combines an ARM-based SoC (System-on-Chip) with a ConnectX-6 Dx network interface controller (NIC).
  - Offers hardware acceleration for networking and storage operations.

- Advanced Networking:
  - Supports up to 200Gb/s Ethernet or InfiniBand networking.
  - Equipped with SR-IOV, RDMA, and GPUDirect Storage capabilities.
    
- Storage Acceleration:
  - Enables offloading for NVMe over Fabrics (NVMe-oF).
  - Provides support for RAID and data mirroring.

- Security Capabilities:
  - Includes a hardware root of trust and secure boot features.
  -  Offers real-time encryption, data isolation, and zero-trust security models.

## Create a slice using BlueField SmartNICs 

This notebook shows how to create an isolated local Ethernet using BlueField Smart NICs and connect compute nodes to it and use FABlib's automatic configuration functionality.


## Import the FABlib Library


In [1]:
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
import ipaddress

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
                     
fablib.show_config();

User: choueiri@email.sc.edu bastion key is valid!
Configuration is valid


Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
Token File,/home/fabric/.tokens.json
Project ID,7bd49888-1cce-4020-82e3-625d8b27f79f
Bastion Host,bastion.fabric-testbed.net
Bastion Username,choueiri_0000118746
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub
Slice Private Key File,/home/fabric/work/fabric_config/slice_key


## Create the Experiment Slice

This example sets up two nodes, each equipped with a BlueField NIC, connected to an isolated local Ethernet. 

Each node is created with a single NIC component, utilizing the `NIC_ConnectX_7_400` and `NIC_Basic` models. These components are attached to the node via PCI passthrough. A list of other available NIC models is provided below. To retrieve the interfaces associated with a NIC component, use the `get_interfaces()` method. Many dedicated NICs feature multiple ports, either of which can be connected to the network.

The node connected to the BlueField SmartNIC runs the `dpu_ubuntu_24` image, which includes DOCA version `2.9.1` by default. Alternatively, users can deploy VMs with the `default_ubuntu_24` image and manually install a different DOCA version as needed.

For automatic configuration, specify a subnet for the network and set the interface mode to `auto` using `iface1.set_mode('auto')` before submitting the request. With this setup, FABlib assigns an IP address from the subnet and configures the device during post-boot setup. Additionally, routes can be pre-configured before submitting the request.

### Available NIC Component Models:
- **NIC_Basic**: 100 Gbps Mellanox ConnectX-6 SR-IOV VF (1 Port)
- **NIC_ConnectX_5**: 25 Gbps Dedicated Mellanox ConnectX-5 PCI Device (2 Ports)
- **NIC_ConnectX_6**: 100 Gbps Dedicated Mellanox ConnectX-6 PCI Device (2 Ports)
- **NIC_ConnectX_7_100**: 100 Gbps Dedicated Mellanox BlueField-3 ConnectX-7 PCI Device (2 Ports)
- **NIC_ConnectX_7_400**: 400 Gbps Dedicated Mellanox BlueField-3 ConnectX-7 PCI Device (2 Ports)
- **NIC_BlueField2_ConnectX_6**: 100 Gbps Dedicated Mellanox BlueField-2 ConnectX-6 PCI Device (2 Ports)

In [2]:
slice_name = 'MySlice2-bluefields'
#site = fablib.get_random_site()
site="RUTG"
print(f"Site: {site}")

node1_name = 'Node1'
node2_name = 'Node2'

network_name='net1'

node1_image = "dpu_ubuntu_24"
node2_image = "default_ubuntu_20"

Site: RUTG


In [3]:
#Create Slice
slice = fablib.new_slice(name=slice_name)

# Network
#net1 = slice.add_l2network(name=network_name, subnet=IPv4Network("192.168.1.0/24"))
net1 = slice.add_l3network(name=network_name)

# Node1
node1 = slice.add_node(name=node1_name, site=site, image=node1_image, cores=4, ram=8, disk=60)
dpu = node1.add_component(model='NIC_ConnectX_7_100', name='nic1')
iface1 =  dpu.get_interfaces()[0]
iface1.set_mode('auto')
net1.add_interface(iface1)

iface2 =  dpu.get_interfaces()[1]
iface2.set_mode('auto')
net1.add_interface(iface2)

# Node2
node2 = slice.add_node(name=node2_name, site=site, image=node2_image, cores=16, ram=32, disk=120)
iface3 = node2.add_component(model='NIC_Basic', name='nic1').get_interfaces()[0]
iface3.set_mode('auto')
net1.add_interface(iface3)


#Submit Slice Request
slice.submit();


Retry: 13, Time: 350 sec


ID,99358c21-7499-4624-8e58-09afb0dcb1d0
Name,MySlice2-bluefields
Lease Expiration (UTC),2026-03-06 15:49:34 +0000
Lease Start (UTC),2026-03-05 15:49:34 +0000
Project ID,7bd49888-1cce-4020-82e3-625d8b27f79f
State,StableOK
Email,CHOUEIRI@email.sc.edu
UserId,78504735-b34f-42a8-be20-8e71d6acf13e


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
4687df42-d70b-48e3-b120-c806ce73cd36,Node1,4,8,100,dpu_ubuntu_24,qcow2,rutg-w3.fabric-testbed.net,RUTG,ubuntu,2620:0:d61:4101:f816:3eff:fe10:d8a7,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2620:0:d61:4101:f816:3eff:fe10:d8a7,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
81afe42f-5c09-41b3-97f9-62827e9f32ef,Node2,16,32,500,default_ubuntu_20,qcow2,rutg-w2.fabric-testbed.net,RUTG,ubuntu,2620:0:d61:4101:f816:3eff:fe71:4a50,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2620:0:d61:4101:f816:3eff:fe71:4a50,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key


ID,Name,Layer,Type,Site,Subnet,Gateway,State,Error
984578e2-45ab-47a0-b59b-2455dd3dd199,net1,L3,FABNetv4,RUTG,10.141.2.0/24,10.141.2.1,Active,


Name,Short Name,Node,Network,Bandwidth,Mode,VLAN,MAC,Physical Device,Device,IP Address,Numa Node,Switch Port
Node1-nic1-p1,p1,Node1,net1,100,auto,,08:9E:49:4F:05:7A,None,None,10.141.2.2,7,HundredGigE0/0/0/14
Node1-nic1-p2,p2,Node1,net1,100,auto,,08:9E:49:4F:05:7B,None,None,10.141.2.4,7,HundredGigE0/0/0/20
Node2-nic1-p1,p1,Node2,net1,100,auto,,26:CA:FA:3F:A0:DB,enp7s0,enp7s0,10.141.2.3,6,HundredGigE0/0/0/7



Time to print interfaces 350 seconds


# Configure the Bluefield Smart NIC

The BlueField SmartNIC is configured by assigning the private IP address `192.168.100.1` to the `tmfifo_net0` device, enabling communication and management of the BlueField DPU. Additionally, the BlueField bundle (BFB) is installed on the DPU via the designated RShim interface, ensuring firmware updates and configuration optimizations for enhanced data center performance.

When using the `dpu_ubuntu_24` image for a node (e.g., `Node1`) connected to the BlueField SmartNIC, the BFB image is available by default at:  
`/opt/bf-bundle/bf-bundle-2.9.1-40_24.11_ubuntu-22.04_prod.bfb`.  
The installation process is initiated when `bluefield.configure()` is executed.

To run custom commands, provide a list of command strings as an argument to `bluefield.configure(commands)`.


In [4]:
slice = fablib.get_slice(slice_name)

node1 = slice.get_node(name=node1_name) 
bluefield = node1.get_component(name='nic1')
output = bluefield.configure()

Checking if local host has root access...
Checking if rshim driver is running locally...
Warn: 'pv' command not found. Continue without showing BFB progress.
Pushing bfb
 INFO[PSC]: PSC BL1 START
 INFO[BL2]: start
 INFO[BL2]: boot mode (rshim)
 INFO[BL2]: VDD_CPU: 810 mV
 INFO[BL2]: VDDQ: 1120 mV
 INFO[BL2]: DDR POST passed
 INFO[BL2]: UEFI loaded
 INFO[BL31]: start
 INFO[BL31]: lifecycle GA Secured
 INFO[BL31]: RPMB Key NOT programmed
 INFO[BL31]: runtime
 INFO[BL31]: MB ping success
 INFO[UEFI]: eMMC init
 INFO[UEFI]: eMMC probed
 INFO[UEFI]: UPVS valid
 INFO[UEFI]: PMI: updates started
 INFO[UEFI]: PMI: total updates: 1
 INFO[UEFI]: PMI: updates completed, status 0
 INFO[UEFI]: PCIe enum start
 INFO[UEFI]: PCIe enum end
 INFO[UEFI]: UEFI Secure Boot (enabled)
 INFO[UEFI]: Redfish enabled
 INFO[UEFI]: exit Boot Service
 INFO[MISC]: Erasing eMMC drive: /dev/mmcblk0
 INFO[MISC]: Erasing NVME drive: /dev/nvme0n1
 INFO[MISC]: Ubuntu installation started
 INFO[MISC]: Installing OS image
 

### Re-apply the network config post re-imaging

Reapply the network configuration on the Node connected to the BlueField. This is only effective when using automatic configuration, where interfaces are set up via `iface1.set_mode('auto')` or `iface1.set_mode('config')`.

In [5]:
# node1.config()
slice.list_interfaces();

Name,Short Name,Node,Network,Bandwidth,Mode,VLAN,MAC,Physical Device,Device,IP Address,Numa Node,Switch Port
Node1-nic1-p1,p1,Node1,net1,100,auto,,08:9E:49:4F:05:7A,None,None,10.141.2.2,7,HundredGigE0/0/0/14
Node1-nic1-p2,p2,Node1,net1,100,auto,,08:9E:49:4F:05:7B,None,None,10.141.2.4,7,HundredGigE0/0/0/20
Node2-nic1-p1,p1,Node2,net1,100,auto,,26:CA:FA:3F:A0:DB,enp7s0,enp7s0,10.141.2.3,6,HundredGigE0/0/0/7


In [6]:
stdout, stderr = node1.execute("sudo ip link set tmfifo_net0 up")

In [15]:
stdout, stderr = node1.execute("sudo ip addr add 192.168.100.1/24 dev tmfifo_net0")

# Accessing DPU:
#### SSH Access:
To SSH into DPU from the VM, use:
```
ssh ubuntu@192.168.100.2
```
After pushing the BFB image to DPU, the default credentials are:  
- **Username:** `ubuntu`  
- **Password:** `ubuntu` (you will be prompted to change it upon first login)
- **New Password:** `passwordpassword`

#### Console Access:
If SSH access is unavailable, you can connect to DPU via the console interface:
```
screen /dev/rshim0/console
```
This method provides an alternative way to manage the SmartNIC if SSH connectivity is lost.


In [8]:
stdout, stderr = node1.execute("ssh-keygen -t ed25519 -f ~/.ssh/id_ed25519 -N ''")

Generating public/private ed25519 key pair.
Your identification has been saved in /home/ubuntu/.ssh/id_ed25519
Your public key has been saved in /home/ubuntu/.ssh/id_ed25519.pub
The key fingerprint is:
SHA256:eNR33jroPDVfUbca5/2D+QhHb2bUJWB8tU5hAh8HzWE ubuntu@Node1
The key's randomart image is:
+--[ED25519 256]--+
|           o+++Eo|
|         . .o.Bo=|
|        . . .ooo=|
|       o   . +o*o|
|      . S    .*o=|
|       .    .o=oo|
|           ..o=Bo|
|           o+o=oo|
|            oo...|
+----[SHA256]-----+


In [11]:
stdout, stderr = node1.execute("sudo ssh-copy-id ubuntu@192.168.100.2")

/usr/bin/ssh-copy-id: ERROR: No identities found


In [12]:
node1.upload_file('scripts/enable_internet_bf.sh', 'enable_internet_bf.sh')
node1.upload_file('scripts/enable_internet_host.sh', 'enable_internet_host.sh')
stdout, stderr = node1.execute("chmod +x enable_internet_bf.sh")
stdout, stderr = node1.execute("chmod +x enable_internet_host.sh")

In [13]:
stdout, stderr = node1.execute("sudo ./enable_internet_host.sh")

[1/5] Enable IPv6 forwarding
[2/5] Add host inside IPv6 (idempotent)
[3/5] Allow forwarding (idempotent)
[4/5] Enable NAT66 (masquerade ULA out WAN) – idempotent
[5/5] Show summary
5: tmfifo_net0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc fq_codel state UNKNOWN group default qlen 1000
    inet6 fd00:1::1/64 scope global tentative 
       valid_lft forever preferred_lft forever
    inet6 fe80::21a:caff:feff:ff02/64 scope link 
       valid_lft forever preferred_lft forever

Chain INPUT (policy ACCEPT 1 packets, 72 bytes)
 pkts bytes target     prot opt in     out     source               destination         
    0     0 ACCEPT     58   --  *      *       ::/0                 ::/0                

Chain FORWARD (policy ACCEPT 0 packets, 0 bytes)
 pkts bytes target     prot opt in     out     source               destination         
    0     0 ACCEPT     0    --  tmfifo_net0 enp1s0  ::/0                 ::/0                
    0     0 ACCEPT     0    --  enp1s0 tmfifo_net0  ::/0

In [14]:
stdout, stderr = node1.execute(
    "ssh ubuntu@192.168.100.2 'sudo apt-get update && sudo apt-get -y install meson ninja-build'"
)

Permission denied, please try again.
Permission denied, please try again.
ubuntu@192.168.100.2: Permission denied (publickey,password).


# Install dependencies host1:

In [ ]:
stdout, stderr = node1.execute(f'sudo apt-get update', quiet = True)

In [ ]:
stdout, stderr = node1.execute(f'sudo apt-get install -y docker.io build-essential python3-pip net-tools', quiet = True)

In [ ]:
stdout, stderr = node1.execute(f'sudo pip3 install meson ninja', quiet = True)

In [ ]:
threads = []

for node in nodes:
    threads.append(node.execute_thread('''
        sudo apt install python3.12-venv
        python3 -m venv .venv
        
        # activate venv
        source .venv/bin/activate
        
        # upgrade pip & install packages
        python3 -m pip install --upgrade pip
        pip install numpy
        pip install pandas
        pip install scikit-learn
        pip install netaddr
        
        # verify numpy version
        python -c "import numpy as np; print('numpy', np.__version__)"
    '''))

for thread in threads:
    thread.result()

# Install dependencies host2:

In [ ]:
stdout, stderr = node2.execute(f'sudo apt-get update', quiet = True)

In [ ]:
stdout, stderr = node2.execute(f'sudo apt-get install -y docker.io build-essential python3-pip net-tools', quiet = True)

In [ ]:
stdout, stderr = node2.execute(f'sudo pip3 install meson ninja', quiet = True)

In [ ]:
threads = []

for server in servers:
    threads.append(server.execute_thread('''
        wget https://content.mellanox.com/ofed/MLNX_OFED-23.07-0.5.0.0/MLNX_OFED_LINUX-23.07-0.5.0.0-ubuntu20.04-x86_64.tgz; 
        tar xvfz MLNX_OFED_LINUX-23.07-0.5.0.0-ubuntu20.04-x86_64.tgz; 
        cd MLNX_OFED_LINUX-23.07-0.5.0.0-ubuntu20.04-x86_64; 
        echo "y" | sudo ./mlnxofedinstall --upstream-libs --dpdk --basic --without-fw-update --enable-sriov --hypervisor
    '''))
    
for thread in threads:
    thread.result()

In [ ]:
threads = []

for server in servers:
    threads.append(server.execute_thread('''
        git clone https://github.com/DPDK/dpdk.git; 
        sudo apt-get install -y build-essential python3-pip python3-pyelftools libnuma-dev pkg-config net-tools hping3;
        cd dpdk;
        sudo meson build;
        cd build;
        sudo ninja;
        sudo ninja install; 
        sudo ldconfig
    '''))

for thread in threads:
    thread.result()

In [ ]:
threads = []

for server in servers:
    threads.append(server.execute_thread('''
        sudo git clone https://github.com/pktgen/Pktgen-DPDK; 
        sudo sed -i \"s/deps += \\[dependency('numa', required: true)\\]/deps += \\[dependency('numa', required: false)\\]/\" /home/ubuntu/Pktgen-DPDK/app/meson.build;
        sudo apt-get install -y cmake libpcap-dev libbsd-dev;
        cd Pktgen-DPDK &&  sudo meson build && sudo ninja -C build && cd build/ && sudo meson install
    '''))

for thread in threads:
    thread.result()

In [17]:
# node2.os_reboot()

In [18]:
slice = fablib.get_slice(slice_name)

node1 = slice.get_node(name=node1_name)        
node2 = slice.get_node(name=node2_name) 

In [19]:
node1.get_ssh_command()

'ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2620:0:d61:4101:f816:3eff:fe10:d8a7'

In [20]:
node2.get_ssh_command()

'ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2620:0:d61:4101:f816:3eff:fe71:4a50'

In [24]:
servers = []
servers.append(slice.get_node(name=node1_name) )     
servers.append(slice.get_node(name=node2_name) )

# Connect DPU to H2

In [47]:
stdout, stderr = node1.execute(
    "ssh ubuntu@192.168.100.2 'sudo ifconfig p1 10.0.0.1/24 up'"
)

In [48]:
stdout, stderr = node1.execute(
    "ssh ubuntu@192.168.100.2 'sudo arp -s 10.0.0.2 26:ca:fa:3f:a0:db'"
)

In [22]:
stdout, stderr = node2.execute("sudo ifconfig enp7s0np0 10.0.0.2/24 up")

In [52]:
stdout, stderr = node2.execute("sudo arp -s 10.0.0.1 e8:9e:49:4f:05:8b")

In [21]:
stdout, stderr = node2.execute(f'sudo ibdev2netdev')
stdout, stderr = node2.execute(f'sudo mst status', quiet=True)
stdout, stderr = node2.execute(f'sudo mst start', quiet=True)
stdout, stderr = node2.execute(f'sudo mst status', quiet=True)

mlx5_0 port 1 ==> enp7s0np0 (Down)


In [ ]:
#number of packets received at p1
ethtool -S p1 | grep -i rx_packets_phy

# Run Pktgen (Node 2)

In [25]:
threads = []

for server in servers:
    threads.append(server.execute_thread(f' sudo sh -c  "echo 4096 > /sys/kernel/mm/hugepages/hugepages-2048kB/nr_hugepages"'))
# for thread in threads:
#     thread.result()

In [ ]:
sudo pktgen -l 0,1 -n 4 -a 07:00.0 -- -P -m "1.0"

# Download Data (Node 2)

In [ ]:
wget https://zenodo.org/records/17165850/files/raw_data.zip?download=1

In [ ]:
mv 'raw_data.zip?download=1' raw_data.zip

In [ ]:
unzip raw_data.zip

# Run DPDK (DPU)

In [ ]:
git clone https://github.com/samiachoueiri/DPDK-ICS-RF.git

In [ ]:
source dpdk_add.sh 

In [ ]:
make

In [ ]:
sudo ./build/ics_app -l 0 -n 4 -a 0000:03:00.1 -- -p 0x1

# Edit and replay pcap

In [ ]:
sudo apt install tcpreplay
cd raw_data/normal/
sudo tcprewrite --enet-smac=02:d0:a8:73:f9:e3 --enet-dmac=e8:9e:49:4e:ff:9b --infile=normal_traffic_2025-01-09_22_43_09.pcap  --outfile=output_benign.pcap
tcpreplay --intf1=enp7s0np0  output_benign.pcap

cd raw_data/fault/
sudo tcprewrite --enet-smac=02:d0:a8:73:f9:e3 --enet-dmac=e8:9e:49:4e:ff:9b --infile=faults_traffic_2025-01-10_03_22_36.pcap  --outfile=output_fault.pcap
sudo tcpreplay --intf1=enp7s0np0  output_fault.pcap

# Call slice

In [1]:
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
import ipaddress

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
                     
slice_name = 'MySlice2-bluefields'
#site = fablib.get_random_site()
# site="KANS"
# print(f"Site: {site}")

node1_name = 'Node1'
node2_name = 'Node2'

network_name='net1'

slice = fablib.get_slice(slice_name)

servers = []
servers.append(slice.get_node(name=node1_name) )     
servers.append(slice.get_node(name=node2_name) )

node1 = servers[0]
node2 = servers[1]

node1 = slice.get_node(name=node1_name)        
node2 = slice.get_node(name=node2_name)

User: choueiri@email.sc.edu bastion key is valid!
Configuration is valid


In [3]:
node1.get_ssh_command()

'ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2620:0:d61:4101:f816:3eff:fe10:d8a7'

In [2]:
node2.get_ssh_command()

'ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2620:0:d61:4101:f816:3eff:fe71:4a50'

# Delete the Slice

Please delete your slice when you are done with your experiment.

In [ ]:
slice = fablib.get_slice(slice_name)
slice.delete()